# CS 4650 — Homework 1: Weather Map/Reduce

This notebook uses MRJob to examine the fixed-width weather records from 1901 and 1902. For each valid wind direction, it reports the lowest temperature, highest temperature, and number of observations. Temperatures are kept in the tenths-of-a-degree units used by the input files.

In [ ]:
from mrjob.job import MRJob
from mrjob.protocol import RawValueProtocol

# The accepted quality codes specified in the assignment.
VALID_QUALITY = {'0', '1', '4', '5', '9'}

class WindDirectionTemperature(MRJob):
    INPUT_PROTOCOL = RawValueProtocol

    def mapper(self, _, line):
        if len(line) < 93:
            return

        wind_direction = line[60:63]
        wind_quality = line[63:64]
        temperature_text = line[87:92]
        temperature_quality = line[92:93]

        if (wind_direction == '999' or
                wind_quality not in VALID_QUALITY or
                temperature_text in {'+9999', '-9999'} or
                temperature_quality not in VALID_QUALITY):
            return

        yield int(wind_direction), {
            'low': int(temperature_text),
            'high': int(temperature_text),
            'count': 1
        }

    def reducer(self, wind_direction, records):
        low = None
        high = None
        count = 0

        for record in records:
            low = record['low'] if low is None else min(low, record['low'])
            high = record['high'] if high is None else max(high, record['high'])
            count += record['count']

        yield wind_direction, {'low': low, 'high': high, 'count': count}


ModuleNotFoundError: No module named 'distutils'

In [ ]:
from pathlib import Path

data_directory = Path('Topic_9_Data (1)') / 'Topic_9_Data'
input_files = [str(data_directory / '1901'), str(data_directory / '1902')]

job = WindDirectionTemperature(args=input_files)
with job.make_runner() as runner:
    runner.run()
    results = []
    for key, value in job.parse_output(runner.cat_output()):
        results.append((key, value))

for wind_direction, summary in sorted(results):
    print(f'{wind_direction}\t{summary}')
